# Supply Chain Optimization - Notebook 01: Data Import & Processing

**Project:** Supply Chain Cost & Operational Efficiency Analysis  
**Dataset:** [Supply Chain DataSet](https://www.kaggle.com/datasets/amirmotefaker/supply-chain-dataset) - [Amir Motefaker](https://www.kaggle.com/amirmotefaker) (Kaggle)  
**Analyst:** *Jakub Siuda*  
**Date:** *28/03/2026*

# Description

This is a dataset collected from a mid-size Fashion and Beauty startup operating a supply chain spanning 100 SKUs across three product categories - Haircare, Skincare, and Cosmetics served by five suppliers, three shipping carriers, and four transportation modes across five Indian cities. 

In this case study I will act as a data analyst that has been tasked with finding ways to optimize costs and operational efficiency of the brand.

# Objectives

* Where is money being lost to inefficient shipping?
* Which suppliers are negatively affecting operational efficiency?
* Are we holding the right inventory for the right products?

**Find ways in which the supply chain is performing inefficiently and recommend solutions on how to optimize costs.**

# Preparations

Dataset used: [Supply Chain DataSet](https://www.kaggle.com/datasets/amirmotefaker/supply-chain-dataset) published by [Amir Motefaker](https://www.kaggle.com/amirmotefaker)

The dataset consists of one clean table featuring 100 beauty and personal care SKUs. The columns include information such as: stock levels, amounts sold, defect rates, etc. This dataframe serves as a perfect candidate for a case study of how a supply chain of a real mid-size brand would operate.

# Processing

For this case study I will be using Python library Pandas to complete my analysis.

While this dataset was uploaded as already cleaned, I will still run a few checks to ensure the integrity of the data.

In [4]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv('data/raw/supply_chain_data.csv')
df_raw.info()

missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
n_dupes = df_raw.duplicated().sum()

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

if missing_report['Missing Count'].sum() == 0:
    print('No missing values detected. Dataset is complete.')

if n_dupes > 0:
    print('WARNING: Duplicate rows found. Review before proceeding.')
    display(df_raw[df_raw.duplicated(keep=False)])
else:
    print('No duplicate rows found.')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product type             100 non-null    object 
 1   SKU                      100 non-null    object 
 2   Price                    100 non-null    float64
 3   Availability             100 non-null    int64  
 4   Number of products sold  100 non-null    int64  
 5   Revenue generated        100 non-null    float64
 6   Customer demographics    100 non-null    object 
 7   Stock levels             100 non-null    int64  
 8   Lead times               100 non-null    int64  
 9   Order quantities         100 non-null    int64  
 10  Shipping times           100 non-null    int64  
 11  Shipping carriers        100 non-null    object 
 12  Shipping costs           100 non-null    float64
 13  Supplier name            100 non-null    object 
 14  Location                 10

During my exploration of the dataset I have confirmed that there are no missing values or duplicate rows present. However there are still a few improvements that I will make to create the best dataset possible for data analysis.

Two columns are named similarly: "Lead times" and "Lead time". These are distinct variables (order procurement lead time vs. supplier-reported lead time) and must be renamed clearly to avoid confusion. Raw column names use mixed case, spaces, and inconsistent pluralisation. All names will be standardised to snake_case. The final “Costs” column will be renamed to “Total supply chain costs” to distinguish it from the “Shipping costs” column.

In [5]:
df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
)

df.rename(columns={
    'lead_times': 'order_lead_time',
    'lead_time':  'supplier_lead_time',
    'costs':      'total_supply_chain_costs'
}, inplace=True)

The next step to take is to ensure that all of the columns have the correct data types. I will cast categorical columns to `pd.Categorical`, confirm numeric columns are float or int and leave the `sku` column as a string, since it is an identifier.

In [6]:
categorical_cols = [
    'product_type',
    'shipping_carriers',
    'supplier_name',
    'location',
    'inspection_results',
    'transportation_modes',
    'routes',
    'customer_demographics'
]

for col in categorical_cols:
    df[col] = pd.Categorical(df[col])

df['inspection_results'] = pd.Categorical(
    df['inspection_results'],
    categories=['Fail', 'Pending', 'Pass'],
    ordered=True
)

numeric_cols = [
    'price', 'availability', 'number_of_products_sold',
    'revenue_generated', 'stock_levels', 'order_lead_time',
    'order_quantities', 'shipping_times', 'shipping_costs',
    'supplier_lead_time', 'production_volumes',
    'manufacturing_lead_time', 'manufacturing_costs',
    'defect_rates', 'total_supply_chain_costs'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

For the last step before the exploratory data analysis, I will create five columns that will help answer the business questions.

| Column | Formula |
|---|---|
| `cost_per_unit_sold` | `shipping_costs / number_of_products_sold` |
| `total_lead_time` | `order_lead_time + shipping_times` |
| `supplier_risk_score` | `(normalised lead time) + (normalised defect rate)` |
| `stock_to_sales_ratio` | `stock_levels / number_of_products_sold` |
| `working_capital_at_risk` | `stock_to_sales_ratio × manufacturing_costs` |

In [7]:
df['cost_per_unit_sold'] = np.where(
    df['number_of_products_sold'] > 0,
    df['shipping_costs'] / df['number_of_products_sold'],
    np.nan
)


df['total_lead_time'] = df['order_lead_time'] + df['shipping_times']


def min_max_scale(series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([0.0] * len(series), index=series.index)
    return (series - min_val) / (max_val - min_val)

df['norm_lead_time']    = min_max_scale(df['total_lead_time'])
df['norm_defect_rate']  = min_max_scale(df['defect_rates'])
df['supplier_risk_score'] = (
    0.5 * df['norm_lead_time'] + 0.5 * df['norm_defect_rate']
).round(4)


df['stock_to_sales_ratio'] = np.where(
    df['number_of_products_sold'] > 0,
    df['stock_levels'] / df['number_of_products_sold'],
    np.nan
).round(4)


df['working_capital_at_risk'] = (
    df['stock_to_sales_ratio'] * df['manufacturing_costs']
).round(2)

Now, all that's left is to save the newly created dataframe and move on to exploratory data analysis in Notebook 02.

In [ ]:
df.to_csv('data/cleaned/supply_chain_clean.csv', index=False)